# Phase 3: Golden Baseline 失敗分析 — 全自動実行

**安全保証:** adapter / training / submission には一切触れません。

## 実行前チェックリスト
- [ ] Accelerator: **GPU T4 x2 または P100**
- [ ] 競技データセット追加済み
- [ ] Nemotron base model 追加済み
- [ ] Golden adapter (huikang/nemotron-adapter) 追加済み

## 実行時間の目安
| ステップ | 時間 |
|---------|------|
| Step 1: カテゴリ分類 | ~5分 |
| Step 2: 推論 | ~60-90分 |
| Step 3: logprob | ~60分 |
| Step 4-7: 集計・分類・レポート | ~5分 |
| **合計** | **~2.5〜3時間** |

---
## Cell 1: リポジトリをclone してスクリプトを取得

In [ ]:
import subprocess, os, sys

REPO_URL    = "https://github.com/hinemos-anzu/Nemotron.git"
REPO_BRANCH = "claude/upbeat-galileo-gqBKp"
REPO_DIR    = "/kaggle/working/Nemotron"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} @ {REPO_BRANCH} ...")
    r = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("ERROR:", r.stderr)
        # --- Internet OFF の場合はここでエラーになります ---
        # その場合は Cell 1b (手動アップロード版) を使ってください
else:
    print(f"既にクローン済み: {REPO_DIR}")

# スクリプトをカレントディレクトリに追加
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\nスクリプト一覧:")
for f in sorted(os.listdir(REPO_DIR)):
    if f.startswith("phase3_") and f.endswith(".py"):
        print(f"  {f}")

---
## Cell 1b: Internet OFF の場合（手動アップロード版）

Internet が OFF の場合は以下を実行してください。
事前に `phase3_*.py` ファイルを Kaggle Dataset としてアップロードしておく必要があります。

In [ ]:
# Internet OFF の場合のみ実行（Cell 1 が成功した場合はスキップ）
import shutil, os, sys, glob

# スクリプトをアップロードした Kaggle Dataset のパスに変更してください
SCRIPTS_DATASET_PATH = "/kaggle/input/phase3-scripts"  # ← 実際のパスに書き換え
REPO_DIR = "/kaggle/working/Nemotron"

if os.path.exists(SCRIPTS_DATASET_PATH):
    os.makedirs(REPO_DIR, exist_ok=True)
    for f in glob.glob(f"{SCRIPTS_DATASET_PATH}/phase3_*.py"):
        dest = f"{REPO_DIR}/{os.path.basename(f)}"
        shutil.copy(f, dest)
        print(f"Copied: {os.path.basename(f)}")
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("\nOK: スクリプト準備完了")
else:
    print(f"ERROR: {SCRIPTS_DATASET_PATH} が見つかりません")
    print("Cell 1 (Internet ON版) を先に実行するか、スクリプトをDatasetとしてアップロードしてください")

---
## Cell 2: パス確認

In [ ]:
import sys, os, glob
sys.path.insert(0, REPO_DIR)
from phase3_run_all import detect_paths

paths = detect_paths()

print("=" * 60)
print("パス自動検出結果")
print("=" * 60)
for k, v in paths.items():
    status = str(v) if v else "NOT FOUND ← 手動で指定してください"
    flag = "✓" if v else "✗"
    print(f"  {flag} {k:12s}: {status}")

# ─────────────────────────────────────────────────
# problems / train_csv が NOT FOUND の場合:
# 下のデバッグ出力で実際のパスを確認 → OVERRIDE_ に設定
# ─────────────────────────────────────────────────
if not paths.get("problems") or not paths.get("train_csv"):
    print("\n--- /kaggle/input/ 以下のファイル一覧 (デバッグ) ---")
    for f in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        if os.path.isfile(f) and not f.endswith((".safetensors", ".bin", ".pt")):
            print(f"  {f}")

# ─────────────────────────────────────────────────
# 競技データの確定パス:
#   /kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv
# ─────────────────────────────────────────────────
OVERRIDE_PROBLEMS = ""   # 例: "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
OVERRIDE_ADAPTER  = ""   # 例: "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
OVERRIDE_MODEL    = ""   # 例: "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

if OVERRIDE_PROBLEMS: paths["problems"] = __import__("pathlib").Path(OVERRIDE_PROBLEMS)
if OVERRIDE_ADAPTER:  paths["adapter"]  = __import__("pathlib").Path(OVERRIDE_ADAPTER)
if OVERRIDE_MODEL:    paths["model"]    = __import__("pathlib").Path(OVERRIDE_MODEL)

missing = [k for k, v in paths.items() if not v]
if missing:
    print(f"\n⚠ 見つからないパス: {missing}")
    print("上の OVERRIDE_ 変数で手動指定してください")
else:
    print("\n✓ 全パス検出済み。次のセルに進めます。")

---
## Cell 3: Dry-run（設定確認のみ、推論なし）

In [ ]:
# ★必ずDry-runを先に実行して設定を確認してください★
import subprocess, sys, os

OUTPUT_DIR = "/kaggle/working/phase3_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [sys.executable, f"{REPO_DIR}/phase3_run_all.py",
       "--dry-run",
       "--output-dir", OUTPUT_DIR,
       *([ "--problems", str(paths["problems"])] if paths["problems"] else []),
       *([ "--adapter",  str(paths["adapter"])]  if paths["adapter"]  else []),
       *([ "--model",    str(paths["model"])]    if paths["model"]    else []),
]

r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

---
## Cell 4: 全ステップ自動実行

⚠ **このセルの実行には約2.5〜3時間かかります。**

Dry-runで問題がなければ実行してください。

In [ ]:
import subprocess, sys, os

OUTPUT_DIR = "/kaggle/working/phase3_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# スモークテスト: 最初は 200問 で動作確認。全件の場合は 0 に変更。
MAX_PROBLEMS = 200  # 0 = 全件 (~2-3時間), 200 = 約10-15分

cmd = [sys.executable, f"{REPO_DIR}/phase3_run_all.py",
       "--steps", "1,2,3,4,5,6,7",
       "--output-dir", OUTPUT_DIR,
       *(["--max-problems", str(MAX_PROBLEMS)] if MAX_PROBLEMS else []),
       *([ "--problems", str(paths["problems"])] if paths["problems"] else []),
       *([ "--adapter",  str(paths["adapter"])]  if paths["adapter"]  else []),
       *([ "--model",    str(paths["model"])]    if paths["model"]    else []),
]

print("実行コマンド:", " ".join(cmd))
print("\n" + "="*60)

# リアルタイムで出力を表示
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print("\n" + "="*60)
print(f"終了コード: {proc.returncode}")

---
## Cell 5: 推論済みの場合（Step 2をスキップ）

既に `golden_validation_predictions.jsonl` がある場合はStep 2をスキップできます。

In [ ]:
# predictions.jsonl が既にある場合はこちらを実行（Step 2をスキップ）
import subprocess, sys, os

OUTPUT_DIR = "/kaggle/working/phase3_analysis"

cmd = [sys.executable, f"{REPO_DIR}/phase3_run_all.py",
       "--skip-inference",
       "--output-dir", OUTPUT_DIR,
       *([ "--adapter",  str(paths["adapter"])]  if paths["adapter"]  else []),
       *([ "--model",    str(paths["model"])]    if paths["model"]    else []),
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

---
## Cell 6: 結果確認

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = "/kaggle/working/phase3_analysis"

print("=" * 60)
print("生成ファイル一覧")
print("=" * 60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f:50s} {size/1024:.1f} KB")

# Golden Baseline accuracy
summary_path = f"{OUTPUT_DIR}/golden_validation_summary.csv"
if os.path.exists(summary_path):
    print("\n=" * 2 + " カテゴリ別 accuracy ")
    df = pd.read_csv(summary_path)
    display(df[df["split"].isin(["overall", "category"])].sort_values("accuracy"))

# 失敗カテゴリ Top10
fail_path = f"{OUTPUT_DIR}/category_failure_summary.csv"
if os.path.exists(fail_path):
    print("\n=" * 2 + " 失敗カテゴリ Priority Top10 ")
    df2 = pd.read_csv(fail_path)
    display(df2.head(10))

# 失敗タイプ
ft_path = f"{OUTPUT_DIR}/failure_type_summary.csv"
if os.path.exists(ft_path):
    print("\n=" * 2 + " 失敗タイプ ")
    df3 = pd.read_csv(ft_path)
    display(df3)

---
## Cell 7: 最終レポート表示

In [ ]:
from IPython.display import Markdown, display

report_path = "/kaggle/working/phase3_analysis/phase3_recommendation.md"
if os.path.exists(report_path):
    with open(report_path, encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("phase3_recommendation.md がまだ生成されていません。Cell 4 を先に実行してください。")

---
## Cell 8: 出力ファイルをKaggle Outputとして保存

Notebook の Output タブからダウンロードできます。
`/kaggle/working/` 以下のファイルは自動的にOutputに含まれます。

In [ ]:
import os, shutil

OUTPUT_DIR = "/kaggle/working/phase3_analysis"

print("=" * 60)
print("Output保存状況確認")
print("=" * 60)
total_size = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    total_size += size
    print(f"  /kaggle/working/phase3_analysis/{f}  ({size/1024:.1f} KB)")

print(f"\n合計: {total_size/1024/1024:.1f} MB")
print("\n✓ Notebookを Save & Run すると Output タブからダウンロードできます。")
print("  重要なファイル:")
print("  - golden_validation_predictions.jsonl  (推論ログ)")
print("  - golden_validation_summary.csv        (accuracy集計)")
print("  - category_failure_summary.csv         (カテゴリ別分析)")
print("  - failure_type_summary.csv             (失敗タイプ集計)")
print("  - phase3_recommendation.md             (最終レポート)")
print("  - run_all.log                          (実行ログ)")